# Project Milestone Two: Modeling and Feature Engineering

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import (
    BaggingRegressor, 
    RandomForestRegressor, 
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

# Progress Tracking
from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [14]:
# =============================================================================
# Prelude: Load the cleaned dataset from Milestone 1 (Parts 3A-3E) and prepare
# for modeling. No feature engineering yet — that comes in Part 2.
# =============================================================================

df_cleaned = pd.read_csv("zillow_cleaned.csv")
print(f"Cleaned dataset: {df_cleaned.shape}")

target = 'taxvaluedollarcnt'

# =============================================================================
# Train/Test Split and Scaling
# =============================================================================
X = df_cleaned.drop(columns=[target])
y = df_cleaned[target]

X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.2, random_state=random_state
)
print(f"\nTrain set: {X_train.shape}, Test set: {X_test.shape}")

# Standardize features using only training data (no data leakage)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
  scaler.fit_transform(X_train),
  columns=X_train.columns,
  index=X_train.index
)
X_test_scaled = pd.DataFrame(
  scaler.transform(X_test),
  columns=X_test.columns,
  index=X_test.index
)

print(f"Scaling complete. Features: {X_train_scaled.shape[1]}")
print(f"Target range: ${y_train.min():,.0f} - ${y_train.max():,.0f}")

Cleaned dataset: (72332, 62)

Train set: (57865, 61), Test set: (14467, 61)
Scaling complete. Features: 61
Target range: $1,000 - $1,112,216


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest
- Gradient Boosting Trees

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [15]:
# =============================================================================
# Part 1: Baseline Setup — Repeated CV and results storage
# =============================================================================
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)
baseline_results = {}

def run_baseline(name, model, X, y, cv):
    """Run repeated CV for a model and store results."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    baseline_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} ± ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

#### Baseline: Ridge Regression

Plain Linear Regression (OLS) produced a mean MAE in the trillions, but investigation showed that 24 of 25 cross-validation folds were normal (`~$149K MAE`) while one fold blew up to `$33` trillion, dragging up the average. This happens because our dataset has columns that measure the same thing (e.g., `calculatedfinishedsquarefeet` and `finishedsquarefeet12` both capture square footage, and `bathroomcnt`, `calculatedbathnbr`, and `fullbathcnt` all count bathrooms). When features are nearly identical, OLS can't figure out how to split credit between them, so the weights swing to extreme values in certain folds.

We attempted to fix this by dropping the near-duplicate columns identified in Milestone 1's 3A Discussion, where we noted these redundancies but deferred removal to compare their behavior in Parts 4-5. However, even after dropping them, OLS still produced blown-up folds (now two instead of one), suggesting subtler collinearity remains among other features in the dataset.

Rather than continuing to hunt down and remove every collinear feature pair which risks losing potentially useful information, we chose Ridge Regression as our linear baseline. Ridge adds an L2 penalty that keeps coefficients small and stable even in the presence of multicollinearity, giving us consistent results across all CV folds without needing to alter our feature set. This makes it a drop-in replacement for OLS that is more robust to the structure of our data.

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

model = LinearRegression()
scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='neg_mean_absolute_error')
print("Per-fold MAE:", -scores)

Per-fold MAE: [1.48051796e+05 1.47844949e+05 1.51253675e+05 1.49480181e+05
 3.34670230e+13 1.48267057e+05 1.49131151e+05 1.50310764e+05
 1.51280926e+05 1.48987680e+05 1.48105463e+05 1.49603898e+05
 1.51447527e+05 1.49021689e+05 1.49818335e+05 1.48018870e+05
 1.50200151e+05 1.51097194e+05 1.47985701e+05 1.50663935e+05
 1.49803149e+05 1.49450689e+05 1.51586664e+05 1.48344671e+05
 1.48919968e+05]


In [17]:
run_baseline('Ridge Regression', Ridge(), X_train_scaled, y_train, cv)

Ridge Regression:
  CV MAE = $149,600.15 ± $1,229.54
  Time: 00:00:01



In [ ]:
# Baseline: Random Forest (~8 min)
# commented out for now since it takes a bit to run and we already ran
# run_baseline('Random Forest', RandomForestRegressor(random_state=random_state), X_train_scaled, y_train, cv)
baseline_results['Random Forest'] = {'mean_mae': 136054.37, 'std_mae': 1070.61, 'time': 393}

In [ ]:
# Baseline: HistGradientBoosting (~20 sec)
run_baseline('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_scaled, y_train, cv)

In [ ]:
# Baseline Results Summary
baseline_df = pd.DataFrame(baseline_results).T
baseline_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
baseline_df['Mean MAE ($)'] = baseline_df['Mean MAE ($)'].map('${:,.2f}'.format)
baseline_df['Std MAE ($)'] = baseline_df['Std MAE ($)'].map('${:,.2f}'.format)
baseline_df['Time (s)'] = baseline_df['Time (s)'].map(format_hms)
print('=== Baseline Results Summary ===')
display(baseline_df)

### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

**Best overall performance:** HistGradientBoosting achieved the lowest CV MAE at `$135,366` (`+-$1,142`), narrowly edging out Random Forest at `$136,054` (`+-$1,071`). Both tree-based ensembles substantially outperformed Ridge Regression (`$149,600` `+-$1,230`), which is expected. Housing prices depend on nonlinear relationships and feature interactions (e.g., square footage × location) that linear models cannot capture.

**Most stable:** Random Forest had the lowest standard deviation (`$1,071`), making it the most consistent across CV folds. This aligns with its design since bagging reduces variance by averaging many decorrelated trees. HistGradientBoosting was nearly as stable (`$1,142`), while Ridge was only slightly higher (`$1,230`). All three models showed strong consistency.

**Overfitting / underfitting:** Ridge Regression shows clear signs of underfitting. It performs `~$14K` worse than the tree models because, as a linear model, it can only learn straight-line relationships between features and price. Housing data often contains nonlinear patterns (e.g., the effect of square footage on price isn't constant across all price ranges), and Ridge simply can't capture those. The tree-based models do not show obvious signs of overfitting at this stage, given their tight standard deviations across folds, though this will be worth monitoring as we add engineered features. It's also worth noting that we originally attempted plain Linear Regression (OLS), which produced MAE in the trillions due to multicollinearity among near-duplicate features. Ridge's L2 regularization was the standard fix for this numerical instability.

**Computational note:** HistGradientBoosting was 24× faster than Random Forest (16 sec vs. 6:38) while achieving slightly better accuracy, thanks to its histogram-based binning approach. This speed advantage will be valuable during hyperparameter tuning in Part 4.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# =============================================================================
# Part 2: Feature Engineering — Add new features from Milestone 1, Part 5
# We add these to the UNSCALED train/test sets, then re-scale.
# =============================================================================

def add_engineered_features(df, zip_stats=None):
  """Add engineered features based on Milestone 1 Part 5 analysis.

  Parameters
  ----------
  df : DataFrame — feature matrix (no target column)
  zip_stats : DataFrame or None — precomputed zip-level stats from training set.
              Pass None when building stats (training), pass stats when applying (test).

  Returns
  -------
  df_new : DataFrame with new features
  zip_stats : DataFrame of zip-level statistics (for reuse on test set)
  """
  df_new = df.copy()

  # 1. Median sqft by zip — "how big are homes in this neighborhood?"
  # Computed ONLY from training data to prevent leakage.
  if zip_stats is None:
      zip_stats = df_new.groupby('regionidzip').agg(
          zip_median_sqft=('calculatedfinishedsquarefeet', 'median'),
      )

  df_new = df_new.merge(zip_stats, on='regionidzip', how='left')
  df_new['zip_median_sqft'] = df_new['zip_median_sqft'].fillna(
      df_new['calculatedfinishedsquarefeet'].median()
  )

  # 2. Bath-to-bed ratio — captures layout balance
  df_new['bath_to_bed_ratio'] = (
      df_new['bathroomcnt'] / df_new['bedroomcnt'].replace(0, np.nan)
  ).fillna(0)

  # 3. Property age — older vs newer construction
  df_new['property_age'] = 2017 - df_new['yearbuilt']
  df_new['property_age'] = df_new['property_age'].fillna(df_new['property_age'].median())

  return df_new, zip_stats

# Apply to train first (computes zip_stats), then test (reuses them)
X_train_eng, zip_stats = add_engineered_features(X_train)
X_test_eng, _ = add_engineered_features(X_test, zip_stats=zip_stats)

new_features = ['zip_median_sqft', 'bath_to_bed_ratio', 'property_age']
print(f"Added {len(new_features)} engineered features: {new_features}")
print(f"Feature count: {X_train.shape[1]} -> {X_train_eng.shape[1]}")

# Re-scale with new features included
scaler_eng = StandardScaler()
X_train_eng_scaled = pd.DataFrame(
  scaler_eng.fit_transform(X_train_eng),
  columns=X_train_eng.columns,
  index=X_train_eng.index
)
X_test_eng_scaled = pd.DataFrame(
  scaler_eng.transform(X_test_eng),
  columns=X_test_eng.columns,
  index=X_test_eng.index
)
print("Scaling complete.")

In [ ]:
# Part 2: CV setup and results storage
eng_results = {}

def run_eng(name, model, X, y, cv):
    """Run repeated CV and store results for Part 2."""
    start = time.time()
    scores = cross_val_score(model, X, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    eng_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name}:")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} \u00b1 ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

In [ ]:
# Part 2: Ridge Regression with engineered features
run_eng('Ridge Regression', Ridge(), X_train_eng_scaled, y_train, cv)

In [ ]:
# Part 2: Random Forest with engineered features
# commented out — already ran, hardcoding results to avoid ~9 min wait
# run_eng('Random Forest', RandomForestRegressor(n_estimators=100, random_state=random_state), X_train_eng_scaled, y_train, cv)
eng_results['Random Forest'] = {'mean_mae': 136026.73, 'std_mae': 1187.38, 'time': 513}

In [ ]:
# Part 2: HistGradientBoosting with engineered features
run_eng('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state), X_train_eng_scaled, y_train, cv)

In [ ]:
# Part 2: Results comparison -- baseline vs. engineered features
eng_df = pd.DataFrame(eng_results).T
eng_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']

print('=== Part 2: Engineered Features Results ===')
for name in eng_results:
    new = eng_results[name]['mean_mae']
    old = baseline_results[name]['mean_mae']
    diff = new - old
    pct = (diff / old) * 100
    arrow = 'v' if diff < 0 else '^'
    print(f"  {name}: ${new:,.2f} (+/-${eng_results[name]['std_mae']:,.2f}) -- {arrow} ${abs(diff):,.2f} ({pct:+.2f}%)")

eng_df['Mean MAE ($)'] = eng_df['Mean MAE ($)'].map('${:,.2f}'.format)
eng_df['Std MAE ($)'] = eng_df['Std MAE ($)'].map('${:,.2f}'.format)
eng_df['Time (s)'] = eng_df['Time (s)'].map(format_hms)
display(eng_df)

### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?




**Notable improvement?** Two of the three models showed improvement from the three engineered features, though the gains were modest. Ridge Regression benefited most, dropping `$513` (-0.34%). HistGradientBoosting improved by `$390` (-0.29%). Random Forest was essentially flat, increasing by just `$48` (+0.04%). The Ridge and HGB improvements are smaller than their respective standard deviations, so they may reflect noise rather than genuine gains.

**Which features helped, and for which models?** The zip-level aggregation feature (`zip_median_sqft`) and the ratio/age features (`bath_to_bed_ratio`, `property_age`) each contribute different types of information. `zip_median_sqft` injects neighborhood-level context that no single row contains — a 2,000 sqft home means something different in a zip where the median is 1,200 sqft vs. 3,000 sqft. `bath_to_bed_ratio` captures layout balance (homes with more bathrooms per bedroom tend to be higher-end), and `property_age` directly encodes construction era effects on value. Ridge benefited the most because these features give the linear model explicit ways to capture location-based and age-based price variation it can't discover on its own. HGB also improved, suggesting that even gradient boosting benefits from pre-computed neighborhood statistics and explicit age features. Random Forest saw almost no change, likely because its random feature subsampling at each split means it doesn't consistently see the new features often enough to benefit.

**Why didn't the features help more?** Two of the three features are derived from columns the models already have access to. `property_age` is a simple transform of `yearbuilt`, and `bath_to_bed_ratio` combines `bathroomcnt` and `bedroomcnt` — tree-based models can already learn these relationships through splits. The zip-level feature provides genuinely new cross-row information by aggregating `calculatedfinishedsquarefeet` by `regionidzip`, but the models were already partially capturing neighborhood effects through the raw zip and geographic features. Features that would likely have more impact are ones the dataset lacks entirely, such as school district ratings, proximity to transit, crime statistics, or walkability scores. Despite the modest results, we retain all three features going forward. They don't hurt, and feature selection in Part 3 will prune any that add noise.

### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [ ]:
# =============================================================================
# Part 3: Feature Selection
# Strategy:
#   - Ridge: SelectKBest (f_regression) — appropriate for linear models
#   - Random Forest: feature importance from a fitted RF
#   - HistGBT: permutation importance from a fitted HGB
# We test multiple k values (top-k features) and pick the best for each model.
# =============================================================================

# --- Step 1: Get feature importances from each method ---

# F-regression scores (for Ridge). A high f-score means there is a statistically significant linear relationship between that feature and the target.
from sklearn.feature_selection import f_regression
from sklearn.inspection import permutation_importance

f_scores, _ = f_regression(X_train_eng_scaled, y_train)
f_score_ranking = pd.Series(f_scores, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# Random Forest feature importance (quick fit with 50 trees)
rf_temp = RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)
rf_temp.fit(X_train_eng_scaled, y_train)
rf_importance = pd.Series(rf_temp.feature_importances_, index=X_train_eng_scaled.columns).sort_values(ascending=False)

# HistGBT feature importance (using permutation importance)
hgb_temp = HistGradientBoostingRegressor(random_state=random_state)
hgb_temp.fit(X_train_eng_scaled, y_train)
perm_result = permutation_importance(hgb_temp, X_train_eng_scaled, y_train,
                                    n_repeats=5, random_state=random_state, n_jobs=-1)
hgb_importance = pd.Series(
  perm_result.importances_mean, index=X_train_eng_scaled.columns
).sort_values(ascending=False)

# Display top 15 features for each method
print("=== Top 15 Features by Method ===\n")
for name, ranking in [('F-score (Ridge)', f_score_ranking),
                     ('Random Forest Importance', rf_importance),
                     ('Histogram Gradient Boosting Importance', hgb_importance)]:
  print(f"{name}:")
  for feat, score in ranking.head(15).items():
      eng_marker = " *ENG*" if feat in new_features else ""
      print(f"  {feat}: {score:.4f}{eng_marker}")
  print()

In [ ]:
# --- Step 2: Sweep top-k features for each model to find best subset ---
# We test k = 5, 10, 15, 20, 30, 45 using a quick 5-fold CV (no repeats) to find
# the best k, then do the full repeated CV on the winner.

k_values = [5, 10, 15, 20, 30, 45]
cv_quick = RepeatedKFold(n_splits=5, n_repeats=1, random_state=random_state)

selection_configs = {
    'Ridge Regression': {'ranking': f_score_ranking, 'model_fn': lambda: Ridge()},
    'Random Forest': {'ranking': rf_importance, 'model_fn': lambda: RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1)},
    'HistGradientBoosting': {'ranking': hgb_importance, 'model_fn': lambda: HistGradientBoostingRegressor(random_state=random_state)},
}

best_k = {}  # store best k and features for each model

for model_name, config in selection_configs.items():
    print(f'{model_name}: sweeping k values...')
    best_mae = float('inf')
    for k in k_values:
        top_features = config['ranking'].head(k).index.tolist()
        X_subset = X_train_eng_scaled[top_features]
        scores = cross_val_score(
            config['model_fn'](), X_subset, y_train,
            cv=cv_quick, scoring='neg_mean_absolute_error', n_jobs=-1
        )
        mae = (-scores).mean()
        marker = '  <-- best' if mae < best_mae else ''
        print(f'  k={k:2d}: MAE = ${mae:,.2f}{marker}')
        if mae < best_mae:
            best_mae = mae
            best_k[model_name] = {'k': k, 'features': top_features, 'mae_quick': mae}
    print(f'  Best: k={best_k[model_name]["k"]}\n')

# Show selected features for each model
for model_name, info in best_k.items():
    eng_count = len([f for f in info['features'] if f in new_features])
    print(f'{model_name} (k={info["k"]}): {eng_count} engineered features retained')
    print(f'  Features: {info["features"]}')
    print()

In [ ]:
# Part 3: Ridge with selected features (full repeated CV)
sel_results = {}

def run_selected(name, model, features, y, cv):
    """Run repeated CV on selected feature subset."""
    start = time.time()
    X_sub = X_train_eng_scaled[features]
    scores = cross_val_score(model, X_sub, y, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1)
    mae_scores = -scores
    elapsed = time.time() - start
    sel_results[name] = {'mean_mae': mae_scores.mean(), 'std_mae': mae_scores.std(), 'time': elapsed}
    print(f"{name} (k={len(features)}):")
    print(f"  CV MAE = ${mae_scores.mean():,.2f} +/- ${mae_scores.std():,.2f}")
    print(f"  Time: {format_hms(elapsed)}\n")

run_selected('Ridge Regression', Ridge(), best_k['Ridge Regression']['features'], y_train, cv)

In [ ]:
# Part 3: Random Forest with selected features (full repeated CV)
# commented out — already ran, hardcoding results to avoid ~8 min wait
# run_selected('Random Forest', RandomForestRegressor(random_state=random_state, n_jobs=-1),
#              best_k['Random Forest']['features'], y_train, cv)
sel_results['Random Forest'] = {'mean_mae': 136094.11, 'std_mae': 1116.45, 'time': 694}

In [ ]:
# Part 3: HistGradientBoosting with selected features (full repeated CV)
run_selected('HistGradientBoosting', HistGradientBoostingRegressor(random_state=random_state),
             best_k['HistGradientBoosting']['features'], y_train, cv)

In [ ]:
# Part 3: Results comparison -- feature selection vs. Part 2 (all engineered features)
print('=== Part 3: Feature Selection Results ===')
for name in sel_results:
    new = sel_results[name]['mean_mae']
    p2 = eng_results[name]['mean_mae']
    p1 = baseline_results[name]['mean_mae']
    diff_p2 = new - p2
    diff_p1 = new - p1
    k = best_k[name]['k']
    print(f"  {name} (k={k}): ${new:,.2f} (+/-${sel_results[name]['std_mae']:,.2f})")
    print(f"    vs Part 2 (65 features): {'+' if diff_p2 > 0 else ''}{diff_p2:,.2f}")
    print(f"    vs Part 1 (61 features): {'+' if diff_p1 > 0 else ''}{diff_p1:,.2f}")

sel_df = pd.DataFrame(sel_results).T
sel_df.columns = ['Mean MAE ($)', 'Std MAE ($)', 'Time (s)']
sel_df['k'] = [best_k[name]['k'] for name in sel_results]
sel_df['Mean MAE ($)'] = sel_df['Mean MAE ($)'].map('${:,.2f}'.format)
sel_df['Std MAE ($)'] = sel_df['Std MAE ($)'].map('${:,.2f}'.format)
sel_df['Time (s)'] = sel_df['Time (s)'].map(format_hms)
display(sel_df)

### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


**Did performance improve after reducing features?**

Feature selection showed mixed results relative to Part 2. Ridge Regression worsened notably (`+$1,431` vs Part 2) and is also worse than the Part 1 baseline (`+$918`), meaning feature selection actually hurt Ridge — likely because trimming from 65 to 45 features removed some of the weaker signals that Ridge's regularization was able to use. Random Forest was essentially flat (`-$8` vs Part 2), and HistGradientBoosting worsened slightly (`+$87` vs Part 2) but still improved `$303` over its Part 1 baseline. The takeaway is that feature selection didn't add value for Ridge or HGB beyond what feature engineering already provided, though HGB's cumulative Part 2+3 improvement over Part 1 held up. Random Forest, which barely changed at any stage, confirms it is largely insensitive to these feature changes.

**Consistently retained features across models:**

Several features appeared in all three models' selected subsets: `calculatedfinishedsquarefeet`, `finishedsquarefeet12`, `bathroomcnt`, `yearbuilt`, `garagetotalsqft`, `bedroomcnt`, `regionidneighborhood`, and the one-hot encoded building quality and heating system indicators. The tree models additionally valued `longitude`, `latitude`, `regionidzip`, and `regionidcity`. These geographic features capture neighborhood-level price differences that F-regression (a linear test) underweights. All three models selected k=45 or k=30 (the highest or near-highest values tested), suggesting the full feature set is largely useful and aggressive pruning hurts more than it helps.

**Were engineered features selected?**

Yes, all three engineered features were retained by all three models. `zip_median_sqft` ranked 6th by F-score for Ridge (3,853) and appeared in the top 10 for Random Forest (importance 0.044) and HGB (importance 0.030), confirming that neighborhood-level size context is a strong price signal. `property_age` ranked 9th for Ridge by F-score (tied with `yearbuilt` at 2,944) and 6th for Random Forest (importance 0.055), its highest ranking across methods. `bath_to_bed_ratio` ranked 12th for Random Forest (importance 0.021) and 12th for HGB (importance 0.006), placing it lower than the other engineered features but still above the selection cutoff for every model. The fact that all three engineered features survived selection across all three models confirms they carry genuinely useful information, even though two of them (`property_age`, `bath_to_bed_ratio`) are derived from columns the models already had access to.

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [ ]:
# =============================================================================
# Part 4: Hyperparameter Tuning
# Each model uses its best feature subset from Part 3.
# =============================================================================

tuned_results = {}

# --- Ridge Regression: GridSearchCV over alpha ---
# Alpha controls regularization strength. Higher = more regularization.
ridge_features = best_k['Ridge Regression']['features']
X_ridge = X_train_eng_scaled[ridge_features]

ridge_params = {'alpha': [0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]}

ridge_grid = GridSearchCV(
    Ridge(), ridge_params, cv=cv,
    scoring='neg_mean_absolute_error', n_jobs=-1
)
start = time.time()
ridge_grid.fit(X_ridge, y_train)
elapsed = time.time() - start

ridge_best = ridge_grid.best_params_
ridge_mae = -ridge_grid.best_score_
ridge_std = ridge_grid.cv_results_['std_test_score'][ridge_grid.best_index_]

tuned_results['Ridge Regression'] = {
    'mean_mae': ridge_mae, 'std_mae': ridge_std,
    'time': elapsed, 'params': ridge_best, 'features': ridge_features
}

print(f'Ridge Regression:')
print(f'  Best params: {ridge_best}')
print(f'  CV MAE = ${ridge_mae:,.2f} +/- ${ridge_std:,.2f}')
print(f'  Time: {format_hms(elapsed)}')

In [ ]:
# --- Random Forest: Two-stage tuning ---
# Stage 1: Quick RandomizedSearchCV (50 trees, 3-fold, 8 combos)
# Stage 2: Full 5x5 CV with best params + 100 trees

rf_features = best_k['Random Forest']['features']
X_rf = X_train_eng_scaled[rf_features]
rf_params = {
  'max_depth': [10, 20, None],
  'max_features': ['sqrt', 0.3],
  'min_samples_split': [2, 5, 10],
}
cv_coarse = RepeatedKFold(n_splits=3, n_repeats=1, random_state=random_state)
start = time.time()
rf_random = RandomizedSearchCV(
  RandomForestRegressor(n_estimators=50, random_state=random_state, n_jobs=-1),
  rf_params, n_iter=8, cv=cv_coarse,
  scoring='neg_mean_absolute_error', random_state=random_state, n_jobs=1
)
rf_random.fit(X_rf, y_train)
best_params = {**rf_random.best_params_, 'n_estimators': 100, 'random_state': random_state}
print(f"Stage 1: {format_hms(time.time() - start)}")
print(f"  Best params: {best_params}")

stage2_start = time.time()
scores = cross_val_score(
  RandomForestRegressor(**best_params, n_jobs=-1),
  X_rf, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
mae_scores = -scores
elapsed = time.time() - start
print(f"Stage 2: {format_hms(time.time() - stage2_start)}")

tuned_results['Random Forest'] = {
  'mean_mae': mae_scores.mean(),
  'std_mae': mae_scores.std(),
  'time': elapsed,
  'params': best_params,
  'features': rf_features
}

print(f"Random Forest:")
print(f"  Best params: {tuned_results['Random Forest']['params']}")
print(f"  CV MAE = ${tuned_results['Random Forest']['mean_mae']:,.2f} +/- ${tuned_results['Random Forest']['std_mae']:,.2f}")

In [ ]:
# --- HistGradientBoosting: Two-stage tuning ---
# Stage 1: Quick search with 3-fold, 12 combos
hgb_features = best_k['HistGradientBoosting']['features']
X_hgb = X_train_eng_scaled[hgb_features]

hgb_params = {
  'learning_rate': [0.01, 0.05, 0.1, 0.2],
  'max_iter': [100, 200, 500],
  'max_depth': [3, 5, 7],
  'min_samples_leaf': [5, 20, 50],
}

cv_coarse = RepeatedKFold(n_splits=3, n_repeats=1, random_state=random_state)
hgb_random = RandomizedSearchCV(
  HistGradientBoostingRegressor(random_state=random_state),
  hgb_params, n_iter=12, cv=cv_coarse,
  scoring='neg_mean_absolute_error', random_state=random_state, n_jobs=-1
)
start = time.time()
hgb_random.fit(X_hgb, y_train)
print(f'Stage 1: {format_hms(time.time() - start)}')
print(f'  Best params: {hgb_random.best_params_}')

# Stage 2: Full CV with best params
best_params = {**hgb_random.best_params_, 'random_state': random_state}
scores = cross_val_score(
  HistGradientBoostingRegressor(**best_params),
  X_hgb, y_train, cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
elapsed = time.time() - start
hgb_mae = (-scores).mean()
hgb_std = (-scores).std()

tuned_results['HistGradientBoosting'] = {
  'mean_mae': hgb_mae, 'std_mae': hgb_std,
  'time': elapsed, 'params': best_params, 'features': hgb_features
}

print(f'Stage 2: {format_hms(elapsed)}')
print(f'  CV MAE = ${hgb_mae:,.2f} +/- ${hgb_std:,.2f}')

In [ ]:
# Part 4: Tuning Results Summary
print('=== Part 4: Tuned Model Results ===')
for name in tuned_results:
    t = tuned_results[name]
    p3 = sel_results[name]['mean_mae']
    p1 = baseline_results[name]['mean_mae']
    diff_p3 = t['mean_mae'] - p3
    diff_p1 = t['mean_mae'] - p1
    print(f"  {name}: ${t['mean_mae']:,.2f} (+/-${t['std_mae']:,.2f})")
    print(f"    Params: {t['params']}")
    print(f"    vs Part 3: {'+' if diff_p3 > 0 else ''}{diff_p3:,.2f}")
    print(f"    vs Part 1 baseline: {'+' if diff_p1 > 0 else ''}{diff_p1:,.2f}")

tuned_df = pd.DataFrame({name: {'Mean MAE ($)': f"${v['mean_mae']:,.2f}",
                                 'Std MAE ($)': f"${v['std_mae']:,.2f}",
                                 'Best Params': str(v['params']),
                                 'Time': format_hms(v['time'])}
                          for name, v in tuned_results.items()}).T
display(tuned_df)

### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


Tuning strategy for each model:
- Ridge Regression: We tested 8 different alpha values using GridSearchCV. The best was 0.01, meaning Ridge barely needed to shrink its coefficients because we already removed redundant features in Part 3. Even with the best alpha, Ridge's MAE only improved by `$0.38`, which tells us the problem isn't alpha. Ridge
is limited because it can only learn straight-line relationships, and housing prices don't follow straight lines.
- Random Forest: We used a two-stage approach. First, a quick RandomizedSearchCV (50 trees, 3-fold, 8 combos) to narrow down the best settings. Then a full 5×5 CV with the winning config. The best params (`max_depth=20`, `max_features='sqrt'`, `min_samples_split=10`) let trees grow deep but require at least 10 samples
to split a node, which prevents overfitting to noise. This improved MAE by `$1,598` over Part 3, bringing RF to `$134,456` which is a substantial gain and its best score across all parts.
- HistGradientBoosting: Same two-stage approach (12 combos coarse, then full CV). The best config (`learning_rate=0.1`, `max_iter=200`, `max_depth=7`, `min_samples_leaf=50`) lets trees grow deeper than the default to capture more complex patterns, while requiring at least 50 samples per leaf to prevent overfitting to noise. HGB improved by `$1,238` over its Part 1 baseline, reaching `$134,128`, the best score of any model in any part.

Did certain preprocessing work better with specific models?

Yes. Feature engineering (Part 2) provided modest but consistent gains for Ridge and HGB, while feature selection (Part 3) confirmed the engineered features were worth keeping. The biggest gains came from hyperparameter tuning (Part 4), especially for Random Forest, which improved by `$1,598`, the largest
single-step improvement of any model in any part. Ridge remained stuck around `$149K` regardless of features or tuning. Its linear assumption is the binding constraint. The tree models benefited most from tuning depth-related parameters (max_depth, min_samples_leaf), which control the bias-variance tradeoff that
feature engineering alone couldn't address. Notably, Random Forest and HGB are now within `$329` of each other (`$134,456` vs `$134,128`), much closer than the `$700` gap at baseline, suggesting that tuning helped RF close the gap by finding a better depth/regularization balance.

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [ ]:
# =============================================================================
# Part 5: Final Model — HistGradientBoosting
# Best performer across all parts. Using tuned params from Part 4
# with the k=45 feature subset from Part 3.
# =============================================================================

# Final model configuration
final_params = {
    'learning_rate': 0.1,
    'max_iter': 200,
    'max_depth': 7,
    'min_samples_leaf': 50,
    'random_state': random_state
}
final_features = best_k['HistGradientBoosting']['features']
final_model = HistGradientBoostingRegressor(**final_params)

print(f'Final Model: HistGradientBoosting')
print(f'Parameters: {final_params}')
print(f'Features: {len(final_features)} selected features')
print()

# --- Repeated CV on training data ---
X_final_train = X_train_eng_scaled[final_features]
X_final_test = X_test_eng_scaled[final_features]

start = time.time()
cv_scores = cross_val_score(
    final_model, X_final_train, y_train,
    cv=cv, scoring='neg_mean_absolute_error', n_jobs=-1
)
cv_elapsed = time.time() - start
cv_mae = (-cv_scores).mean()
cv_std = (-cv_scores).std()

print(f'=== Cross-Validation Results (5-fold, 5 repeats) ===')
print(f'  CV MAE = ${cv_mae:,.2f} +/- ${cv_std:,.2f}')
print(f'  Time: {format_hms(cv_elapsed)}')
print()

# --- Test set evaluation ---
final_model.fit(X_final_train, y_train)
y_pred = final_model.predict(X_final_test)
test_mae = mean_absolute_error(y_test, y_pred)

print(f'=== Held-Out Test Set Results ===')
print(f'  Test MAE = ${test_mae:,.2f}')
print()

# --- Summary comparison across all parts ---
print(f'=== Performance Progression ===')
print(f'  Part 1 (baseline):          ${baseline_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 2 (+ eng features):    ${eng_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 3 (feature selection):  ${sel_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 4 (tuned):             ${tuned_results["HistGradientBoosting"]["mean_mae"]:,.2f}')
print(f'  Part 5 (final CV):          ${cv_mae:,.2f}')
print(f'  Part 5 (test set):          ${test_mae:,.2f}')
print(f'\n  Total improvement: ${baseline_results["HistGradientBoosting"]["mean_mae"] - cv_mae:,.2f} ({(baseline_results["HistGradientBoosting"]["mean_mae"] - cv_mae) / baseline_results["HistGradientBoosting"]["mean_mae"] * 100:.2f}%)')

### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

1. Model Selection

We picked HistGradientBoosting because it had the lowest MAE at every stage, from Part 1's baseline (`$135,366`) to Part 4's tuned result (`$134,128`). On the held-out test set it got `$135,386`, which is `$1,259` off from the CV estimate. This is a slightly larger gap than ideal, but still reasonable and not indicative of serious overfitting.

HGB beat Random Forest by `~$329` after tuning (down from `~$700` at baseline) and Ridge by `~$14,700`. It was also much faster, 14 seconds vs ~8 minutes for Random Forest.

The downside is interpretability. With Ridge, you can say "each extra square foot adds `$X` to the price." With HGB you can't, it's a black box. But we can still check which features matter most using permutation importance, and the `~$15K` accuracy advantage over Ridge is worth the trade-off for predicting home prices.

2. Revisiting an Early Decision

In Milestone 1, we imputed missing values for `airconditioningtypeid` (68% missing) and `heatingorsystemtypeid` (36% missing) using mode imputation, then one-hot encoded them. The idea was that AC and heating type could signal home quality. A home with central AC is likely worth more than one without.

Some of these one-hot columns (`heatingorsystemtypeid_2.0`, `heatingorsystemtypeid_7.0`, `buildingqualitytypeid_9.0`) did show up in feature importance rankings and survived feature selection. But with 68% of AC values missing, mode imputation just assigned most homes the same value, making it less useful. A simple binary "has_ac" feature (yes/no) probably would have worked better than encoding the specific AC type, since presence vs. absence of AC is the real signal. We kept the current approach because feature selection in Part 3 handled it, low-value columns got pruned and informative ones were kept. If we were to redo this, we would replace the one-hot encoded AC columns with a single binary `has_ac` feature, which would reduce dimensionality while preserving the meaningful signal (presence vs. absence of AC).

3. Lessons Learned

The biggest lesson was that cross-row aggregation features (Part 2) provided more value than simple row-level transforms would have. The zip-level statistic `zip_median_sqft` gave every model neighborhood context that no single property record contains, while `bath_to_bed_ratio` and `property_age` provided intuitive, domain-motivated signals that all three engineered features
survived feature selection. That said, the largest single improvement still came from hyperparameter tuning (Part 4), which improved HGB by `$1,238` over baseline and brought Random Forest within `$329` of HGB. This shows that for tree-based models, tuning depth and regularization parameters remains the highest-leverage step.

With more time or data, we would explore:

(1) external data like school ratings, crime stats, and walkability scores, genuinely new information the model can't derive from existing features.

(2) target encoding for high-cardinality columns like `regionidzip` and `regionidcity` instead of treating them as raw numbers.

(3) stacking the three models together, since Ridge captures different patterns than the tree models and could complement them.